# ResNet50 & EfficientNetV2 - Exploratory Experiments

This notebook is for **interactive exploration** of Models 2 and 3
(ResNet50, EfficientNetV2) - visualizing data, running a few quick epochs,
and plotting results. For the real, full training run that produces the
final `results.json` used in the report, run the command-line scripts
instead:

```bash
python -m models.resnet50.train
python -m models.efficientnetv2.train
```

Run this notebook from the project root (so relative imports/paths work).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # so `src` and `models` import correctly when run from notebooks/

import torch
import matplotlib.pyplot as plt

from src.utils import set_seed, load_config, get_device, count_parameters
from src.data_pipeline import build_dataloaders
from src.metrics import evaluate_model

set_seed(42)
device = get_device()
print("Using device:", device)

## 1. Load data and preview a few samples

In [ ]:
resnet_cfg = load_config("../configs/resnet50.yaml")
loaders = build_dataloaders(resnet_cfg["data"])

print("Train batches:", len(loaders["train"]))
print("Val batches:", len(loaders["val"]))
print("Test batches:", len(loaders["test"]))
print("Cross-generator batches:", len(loaders["cross_gen"]))

In [ ]:
# Preview a batch of images with their labels (0=real, 1=fake)
images, labels = next(iter(loaders["train"]))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i, ax in enumerate(axes.flat):
    if i >= len(images):
        break
    img = images[i] * std + mean  # undo normalization for display
    img = img.permute(1, 2, 0).clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title("fake" if labels[i].item() == 1 else "real")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Quick ResNet50 sanity-check run

A short run (a few epochs, base frozen) just to confirm the pipeline works
and see an early learning curve - NOT the full training run for your report.

In [ ]:
from models.resnet50.model import get_model as get_resnet, freeze_backbone

resnet = get_resnet(num_classes=2).to(device)
freeze_backbone(resnet)
print(f"Trainable parameters (head only): {count_parameters(resnet):,}")

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-3
)
criterion = torch.nn.CrossEntropyLoss()

quick_epochs = 2  # just for a fast sanity check in this notebook
history = {"train_loss": [], "val_loss": []}

for epoch in range(quick_epochs):
    resnet.train()
    running_loss = 0.0
    for images, labels in loaders["train"]:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = resnet(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(loaders["train"].dataset)

    resnet.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in loaders["val"]:
            images, labels = images.to(device), labels.to(device)
            logits = resnet(images)
            val_loss += criterion(logits, labels).item() * images.size(0)
    val_loss /= len(loaders["val"].dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch+1}/{quick_epochs} - train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

In [ ]:
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 quick sanity-check run")
plt.legend()
plt.show()

## 3. Evaluate the quick run on test + cross-generator set

Just to see the metrics function works end-to-end - accuracy here will be
low since this was only a 2-epoch sanity check, not the real training run.

In [ ]:
test_metrics = evaluate_model(resnet, loaders["test"], device)
cross_gen_metrics = evaluate_model(resnet, loaders["cross_gen"], device)

print("Test accuracy:", round(test_metrics["accuracy"], 4))
print("Cross-generator accuracy:", round(cross_gen_metrics["accuracy"], 4))
print("Test confusion matrix:", test_metrics["confusion_matrix"])

## 4. EfficientNetV2 - same quick sanity check

Repeat the same short experiment with EfficientNetV2 for a side-by-side
early comparison before committing to the full training runs.

In [ ]:
from models.efficientnetv2.model import get_model as get_efficientnet, freeze_backbone as freeze_efficientnet

efficientnet = get_efficientnet(num_classes=2).to(device)
freeze_efficientnet(efficientnet)
print(f"Trainable parameters (head only): {count_parameters(efficientnet):,}")

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()), lr=1e-3
)

eff_history = {"train_loss": [], "val_loss": []}

for epoch in range(quick_epochs):
    efficientnet.train()
    running_loss = 0.0
    for images, labels in loaders["train"]:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = efficientnet(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(loaders["train"].dataset)

    efficientnet.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in loaders["val"]:
            images, labels = images.to(device), labels.to(device)
            logits = efficientnet(images)
            val_loss += criterion(logits, labels).item() * images.size(0)
    val_loss /= len(loaders["val"].dataset)

    eff_history["train_loss"].append(train_loss)
    eff_history["val_loss"].append(val_loss)
    print(f"Epoch {epoch+1}/{quick_epochs} - train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

In [ ]:
plt.plot(history["train_loss"], label="ResNet50 train loss")
plt.plot(eff_history["train_loss"], label="EfficientNetV2 train loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Quick sanity-check comparison (NOT final results)")
plt.legend()
plt.show()

## Next steps

Once you're happy the pipeline works (as shown above), run the real,
full training for both models from the terminal instead of this notebook:

```bash
python -m models.resnet50.train
python -m models.efficientnetv2.train
```

Then use `compare_models.py` at the project root to generate the final
comparison table and plots for your report.